# Union and Dedup Notebook

This notebook consumes dataset-level standardized TSV outputs and performs the final corpus assembly stage: **union + deduplication + export**.

## Expected upstream inputs
- `outputs/preprocessing/01_hatexplain_standardized.tsv`
- `outputs/preprocessing/02_mhs_standardized.tsv`
- `outputs/preprocessing/03_elsherief_standardized.tsv` (optional, controlled by config)

## What this notebook does
1. Loads standardized per-dataset outputs.
2. Ensures required columns are present and normalized.
3. Unions datasets into a single table.
4. Deduplicates with ID-first strategy, then text-key fallback.
5. Saves final outputs (`04_union_primary.tsv`, `04_dedup_primary.tsv`, and `04_union_dedup_summary.tsv`).

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List
import re

import pandas as pd

# Resolve workspace root robustly for notebook execution in multiple contexts.
WORKDIR = Path.cwd()
if not (WORKDIR / 'outputs').exists():
    WORKDIR = Path('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles')

@dataclass
class UnionConfig:
    # Include or exclude ElSherief in final union.
    include_elsherief: bool = True

    # Input standardized files from the dataset-specific notebooks.
    hatexplain_tsv: Path = WORKDIR / 'outputs' / 'preprocessing' / '01_hatexplain_standardized.tsv'
    mhs_tsv: Path = WORKDIR / 'outputs' / 'preprocessing' / '02_mhs_standardized.tsv'
    elsherief_tsv: Path = WORKDIR / 'outputs' / 'preprocessing' / '03_elsherief_standardized.tsv'

    # Final union/dedup outputs.
    union_output: Path = WORKDIR / 'outputs' / 'unioned_data' / '04_union_primary.tsv'
    dedup_output: Path = WORKDIR / 'outputs' / 'unioned_data' / '04_dedup_primary.tsv'
    summary_output: Path = WORKDIR / 'outputs' / 'unioned_data' / '04_union_dedup_summary.tsv'

cfg = UnionConfig()
cfg.union_output.parent.mkdir(parents=True, exist_ok=True)
cfg

UnionConfig(include_elsherief=True, hatexplain_tsv=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing/01_hatexplain_standardized.tsv'), mhs_tsv=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing/02_mhs_standardized.tsv'), elsherief_tsv=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/preprocessing/03_elsherief_standardized.tsv'), union_output=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/unioned_data/04_union_primary.tsv'), dedup_output=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/unioned_data/04_dedup_primary.tsv'), summary_output=PosixPath('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/unioned_data/04_union_dedup_summary.tsv'))

In [2]:
REQUIRED_COLUMNS = [
    'post_id',
    'text',
    'raw_label',
    'binary_hate',
    'targets',
    'dataset',
    'text_dedup_key',
]


def normalize_text_for_dedup(text: str) -> str:
    """Create a lowercase, whitespace-normalized text key for dedup fallback."""
    if not isinstance(text, str):
        return ''
    return re.sub(r'\s+', ' ', text.strip().lower())


def read_standardized_tsv(path: Path, dataset_name: str) -> pd.DataFrame:
    """Load one standardized TSV and enforce minimum schema guarantees."""
    if not path.exists():
        raise FileNotFoundError(f'Missing standardized file for {dataset_name}: {path}')

    df = pd.read_csv(path, sep='\t')

    # Ensure required columns exist even if an upstream export omitted one accidentally.
    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = ''

    # Keep only the shared core schema plus optional metadata columns that may be useful later.
    keep_columns = REQUIRED_COLUMNS + [col for col in ['n_annotations'] if col in df.columns]
    df = df[keep_columns].copy()

    # Standardize dtypes and ensure a dedup key always exists.
    df['post_id'] = df['post_id'].astype(str).replace('nan', '')
    df['text'] = df['text'].astype(str)
    df['dataset'] = dataset_name

    missing_key = df['text_dedup_key'].isna() | (df['text_dedup_key'].astype(str).str.strip() == '')
    if missing_key.any():
        df.loc[missing_key, 'text_dedup_key'] = df.loc[missing_key, 'text'].apply(normalize_text_for_dedup)

    # Binary label should remain nullable integer where possible.
    df['binary_hate'] = pd.to_numeric(df['binary_hate'], errors='coerce').astype('Int64')

    return df


def deduplicate_union(union_df: pd.DataFrame) -> pd.DataFrame:
    """Apply ID-first dedup, then fallback text-key dedup, matching original project behavior."""
    # Pass 1: keep first row for every explicit post ID.
    with_id = union_df[union_df['post_id'].notna() & (union_df['post_id'].astype(str).str.strip() != '')]
    with_id = with_id.drop_duplicates(subset=['post_id'], keep='first')

    # Pass 2: among rows not retained in pass 1, dedup by normalized text key.
    without_id = union_df[~union_df.index.isin(with_id.index)]
    without_id = without_id.drop_duplicates(subset=['text_dedup_key'], keep='first')

    # Final pass: enforce one row per text key globally to remove cross-source repeats.
    dedup_df = pd.concat([with_id, without_id], ignore_index=True)
    dedup_df = dedup_df.drop_duplicates(subset=['text_dedup_key'], keep='first')

    return dedup_df

In [3]:
# Load mandatory standardized datasets.
hx_std = read_standardized_tsv(cfg.hatexplain_tsv, dataset_name='hatexplain')
mhs_std = read_standardized_tsv(cfg.mhs_tsv, dataset_name='mhs')

parts: List[pd.DataFrame] = [hx_std, mhs_std]

# Optionally include ElSherief if configured and available.
if cfg.include_elsherief:
    if cfg.elsherief_tsv.exists():
        els_std = read_standardized_tsv(cfg.elsherief_tsv, dataset_name='elsherief')
        parts.append(els_std)
    else:
        print('ElSherief inclusion is enabled, but standardized file is missing; continuing without it.')

# Union all selected standardized datasets.
union_df = pd.concat(parts, ignore_index=True)

# Deduplicate union output using the ID-first + text-key fallback strategy.
dedup_df = deduplicate_union(union_df)

# Build a compact summary table that makes run-to-run checks easy.
summary_df = pd.DataFrame([
    {'metric': 'hatexplain_rows', 'value': len(hx_std)},
    {'metric': 'mhs_rows', 'value': len(mhs_std)},
    {'metric': 'elsherief_rows_included', 'value': int((dedup_df['dataset'] == 'elsherief').sum()) if 'dataset' in dedup_df.columns else 0},
    {'metric': 'union_rows', 'value': len(union_df)},
    {'metric': 'dedup_rows', 'value': len(dedup_df)},
])

# Persist final outputs for downstream audit notebooks/scripts.
union_df.to_csv(cfg.union_output, sep='\t', index=False)
dedup_df.to_csv(cfg.dedup_output, sep='\t', index=False)
summary_df.to_csv(cfg.summary_output, sep='\t', index=False)

print('Saved union output to:', cfg.union_output)
print('Saved dedup output to:', cfg.dedup_output)
print('Saved summary output to:', cfg.summary_output)
display(summary_df)
display(dedup_df.head(5))

Saved union output to: /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/unioned_data/04_union_primary.tsv
Saved dedup output to: /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/unioned_data/04_dedup_primary.tsv
Saved summary output to: /Users/RevaH/Documents/COS534/benchmarking_dogwhistles/outputs/unioned_data/04_union_dedup_summary.tsv


,metric,value
0,hatexplain_rows,20148
1,mhs_rows,39565
2,elsherief_rows_included,21472
3,union_rows,81193
4,dedup_rows,81093


,post_id,text,raw_label,binary_hate,targets,dataset,text_dedup_key,n_annotations
0,1179055004553900032_twitter,i dont think im getting my baby them white 9 h...,"[{""label"": ""normal"", ""annotator_id"": 1, ""targe...",0,NaN,hatexplain,i dont think im getting my baby them white 9 h...,NaN
1,1179063826874032128_twitter,we cannot continue calling ourselves feminists...,"[{""label"": ""normal"", ""annotator_id"": 1, ""targe...",0,NaN,hatexplain,we cannot continue calling ourselves feminists...,NaN
2,1178793830532956161_twitter,nawt yall niggers ignoring me,"[{""label"": ""normal"", ""annotator_id"": 4, ""targe...",0,african,hatexplain,nawt yall niggers ignoring me,NaN
3,1179088797964763136_twitter,<user> i am bit confused coz chinese ppl can n...,"[{""label"": ""hatespeech"", ""annotator_id"": 1, ""t...",1,asian,hatexplain,<user> i am bit confused coz chinese ppl can n...,NaN
4,1179085312976445440_twitter,this bitch in whataburger eating a burger with...,"[{""label"": ""hatespeech"", ""annotator_id"": 4, ""t...",1,caucasian|women,hatexplain,this bitch in whataburger eating a burger with...,NaN


## Next Step: Target Label Analysis

Target-label standardization, groupwise label analysis, and post-union low-group dropping have been moved to:

- `data_preprocessing/05_target_label_analysis_and_filtering.ipynb`

This notebook now focuses only on union + dedup output generation.